# Practice: Encode One Page, and Argue With Yourself

**Computational Text Analysis Pack · Module 20 — Encoding as Method**

*Technique:* A full TEI encoding decision log, and querying what you asserted

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Sagaustus/adh-dh-notebooks/blob/main/09-encoding-as-method/20d_practice_encode_a_page.ipynb)

Data: `colonial_texts.csv` — from the `data/` folder of this pack.

---

In [ ]:
# Setup — run this first.
import os, warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
from collections import Counter

# The data this notebook needs. It is downloaded from CourseHub automatically;
# uploading by hand is only a fallback if that server is unreachable.
NEEDED = ['colonial_texts.csv']
DATA_URL = 'https://coursehub.africandigitalhumanities.ca/ide/pack-data/text/'

def _missing():
    return [f for f in NEEDED if not os.path.exists(f)]

missing = _missing()
if missing:
    # Fetch from CourseHub first, so nobody has to upload anything.
    import urllib.request, urllib.parse
    for _f in list(missing):
        try:
            urllib.request.urlretrieve(DATA_URL + urllib.parse.quote(_f), _f)
            print(f'downloaded {_f}')
        except Exception as _e:
            print(f'could not fetch {_f}: {_e}')
    missing = _missing()

if missing:
    try:
        from google.colab import files
    except ImportError:
        raise SystemExit('Place these next to the notebook: ' + ', '.join(missing))
    # Ask again until everything has arrived. The upload widget returns as soon
    # as you close it, so picking only some of the files would otherwise fail a
    # few lines below with a confusing FileNotFoundError.
    for _ in range(4):
        print('Select ALL of these at once (ctrl-click / cmd-click to multi-select):')
        print('   ' + ', '.join(missing))
        files.upload()
        missing = _missing()
        if not missing:
            break
        print('Still needed: ' + ', '.join(missing))
    if missing:
        raise SystemExit(
            'Missing: ' + ', '.join(missing) + '. Re-run this cell and select '
            'every file listed, or upload them with the folder icon on the left.')

df_colonial = pd.read_csv('colonial_texts.csv')
NIGERIAN = ['Hausa', 'Igbo', 'Yoruba', 'Nigerian Pidgin']
# Most of this pack runs on scikit-learn and the standard library. Also
# available: nltk, spacy, gensim and networkx (modules 14-19), Keras
# (case studies), and a GPU pod reached through /ide/api/neural/ for the
# neural modules — never called directly, so no token lives in a notebook.

print(f'Loaded. df_colonial has {len(df_colonial):,} rows and {df_colonial.shape[1]} columns.')

## The question

> **Take one page of an archival text and encode it properly. What did you have
> to decide, and what would a different scholar have decided?**

Module 20 argued that encoding is interpretation. This is that claim as an
exercise: you will make roughly a dozen judgement calls on a single page, and
most of them are contestable.

### How to use this notebook

A worked case, not a tutorial. Run it once for the shape of the argument, then
change the question at the top to one of your own.

In [ ]:
import re

def tokenise(text):
    """Words, lowercased, Unicode-aware.

    `\\w` matches Unicode letters, so ọ, à, ƙ, ɗ and ɓ survive. It does not
    match combining marks, and a tone mark on ọ or ẹ is always one, so the
    pattern adds the combining-accent range U+0300-U+036F; without it ọ̀rọ̀
    splits into ọ + rọ. The common shortcut `[a-z]+` throws away far more and
    quietly changes every count in this notebook — see notebook 03b.
    """
    return re.findall(r"[\w\u0300-\u036f]+", str(text).lower())

sample = "Àwọn ọmọ ìlú ti ṣe ìpàdé lórí ọ̀rọ̀ ẹ̀kọ́ — ƙungiyar ɗalibai ta yi taro"
print("sample     :", sample)
print("tokens     :", tokenise(sample))
print("bare \\w+   :", re.findall(r"\w+", sample.lower()))
print("naive [a-z]:", re.findall(r"[a-z]+", sample.lower()))
print()
print("The naive pattern splits words at every accented letter; bare \\w+")
print("splits them wherever a tone mark sits on ọ or ẹ.")

In [ ]:
import re
from xml.sax.saxutils import escape
import xml.etree.ElementTree as ET

TEI_NS = "http://www.tei-c.org/ns/1.0"
ns = {"t": TEI_NS}
page = df_colonial.iloc[7]
text = " ".join(str(page["text"]).split())[:900]
print(f"source: {page['title'][:52]} / {page['author']}\n")
print(text[:400], "...")

In [ ]:
# The decision log. Each entry is a choice, with the alternative stated.
DECISIONS = [
 ("persName", r"\b((?:Mr|Mrs|Dr|Sir|Lord|Captain|Chief|King)\.?\s+[A-Z]\w+)",
  "title + capitalised word", "ALTERNATIVE: tag every capitalised word (higher recall, many false positives)"),
 ("placeName", r"\b(?:at|in|near|to)\s+([A-Z][a-z]{3,})",
  "locative preposition + capital", "ALTERNATIVE: use a gazetteer of known places (higher precision, misses the unlisted)"),
 ("date", r"\b(1[6-9]\d{2})\b",
  "four-digit years 1600-1999", "ALTERNATIVE: also tag 'last spring', 'the following year' (needs human judgement)"),
 ("num", r"\b(\d{1,3}(?:,\d{3})+|\d+)\b",
  "any numeral", "ALTERNATIVE: distinguish quantities from page refs and dates"),
]
for tag, pat, why, alt in DECISIONS:
    n = len(re.findall(pat, text))
    print(f"<{tag}>  {n:>3} matches   rule: {why}")
    print(f"          {alt}\n")

In [ ]:
body = escape(text)
for tag, pat, _, _ in DECISIONS:
    body = re.sub(pat, lambda m, t=tag: f"<{t}>{m.group(1)}</{t}>", body)
tei = f'''<TEI xmlns="{TEI_NS}">
  <teiHeader><fileDesc>
    <titleStmt><title>{escape(str(page["title"]))}</title>
      <author>{escape(str(page["author"]))}</author></titleStmt>
    <publicationStmt><p>Public domain, Project Gutenberg.</p></publicationStmt>
    <sourceDesc><p>Passage {page["work_id"]}, ADH colonial-discourse corpus.</p></sourceDesc>
  </fileDesc>
  <encodingDesc><p>Names, places, dates and numerals tagged by regular
    expression; rules and their alternatives are logged with this file.
    Coverage is partial and biased toward English orthography.</p></encodingDesc>
  </teiHeader>
  <text><body><p>{body}</p></body></text>
</TEI>'''
root = ET.fromstring(tei)
print("what this encoding ASSERTS:\n")
for tag in ["persName", "placeName", "date", "num"]:
    found = [e.text for e in root.findall(f".//t:{tag}", ns) if e.text]
    print(f"   <{tag}:>  {found[:8]}")

In [ ]:
# Now the audit: what did you assert that is wrong, and what did you miss?
all_caps = set(re.findall(r"\b[A-Z][a-z]{3,}\b", text))
tagged = set()
for tag in ["persName", "placeName"]:
    for e in root.findall(f".//t:{tag}", ns):
        if e.text: tagged.update(e.text.split())
missed = sorted(all_caps - tagged)
print(f"capitalised words NOT tagged as a name or place ({len(missed)}):")
print("   " + ", ".join(missed[:16]))
print()
print("Go through that list. Some are sentence-initial ordinary words (correct to")
print("skip), some are months or ships (a different tag), and some are names your")
print("rules missed. Each one you reclassify is an interpretive act, and TEI's")
print("job is to record it where a reader can dispute it.")

## What this licenses

**Defensible:** "The passage was encoded in TEI P5 with `<persName>`,
`<placeName>`, `<date>` and `<num>`; rules and rejected alternatives are logged
in `<encodingDesc>`; recall against capitalised tokens is [X]."

**Not defensible:** a queried count from an encoding whose rules were not
published. Module 20b's point stands: a structural query gives an exact answer
about your *markup*, and a reader who mistakes that for an answer about the text
has been misled by your silence.

**The alternative column is the real deliverable.** An encoding that records only
what was chosen, and not what was rejected, has hidden the argument.

### 🔧 Your turn 1

Encode the same passage a second time with the ALTERNATIVE rule for `persName`, and count how many entities differ.

<details markdown="1">
<summary><b>What you should have seen</b> — click to expand</summary>

The two encodings typically disagree on a third to a half of the entities, which
is the same magnitude module 15d found for independently built dictionaries and
module 17 found for independent coders. The consistency of that figure across
three quite different tasks is the deep point of this course: wherever human
judgement enters a pipeline, expect roughly half agreement unless you have
measured and improved it.

</details>